# 02 — Preprocesamiento

Estabilización (Farnebäck + RANSAC), corrección de iluminación (CLAHE),
detección de artefactos y cálculo de IQS.

In [ ]:
import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH  = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
PROC_PATH  = DATA_PATH / "processed"
PROC_PATH.mkdir(exist_ok=True)


## 2.1 Estabilización de video — Farnebäck + RANSAC

In [ ]:
def stabilize_frames(frames: list[np.ndarray]) -> list[np.ndarray]:
    """Elimina movimiento global con flujo óptico denso + RANSAC."""
    ref_gray  = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    stabilized = [frames[0]]

    for frame in frames[1:]:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            ref_gray, gray, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )
        h, w = gray.shape
        y, x  = np.mgrid[0:h, 0:w]
        src   = np.column_stack([x.ravel(), y.ravel()]).astype(np.float32)
        dst   = src + flow.reshape(-1, 2)
        # RANSAC cada 8 px para velocidad
        M, _ = cv2.estimateAffinePartial2D(
            src[::8], dst[::8], method=cv2.RANSAC, ransacReprojThreshold=3
        )
        if M is not None:
            stabilized.append(cv2.warpAffine(frame, M, (w, h)))
        else:
            stabilized.append(frame)
    return stabilized


## 2.2 Corrección de iluminación — CLAHE

In [ ]:
def correct_illumination(frame: np.ndarray, clip_limit=2.0, tile=(8,8)) -> np.ndarray:
    """CLAHE en canal L del espacio LAB para preservar color."""
    lab  = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    l_eq   = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)


## 2.3 Score de calidad — IQS automático

In [ ]:
def compute_iqs(frame: np.ndarray) -> dict:
    """
    Image Quality Score basado en:
    - Sharpness: varianza del Laplaciano (blur detector)
    - Brightness: media de intensidad (0=negro, 255=blanco)
    - Contrast: desviación estándar de intensidad
    Retorna dict con valores y flag 'quality_ok'.
    """
    gray       = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness  = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = float(gray.mean())
    contrast   = float(gray.std())

    # Umbrales empíricos — ajustar con tus videos
    quality_ok = (
        sharpness  > 50   and   # frame nítido
        10 < brightness < 230   and   # ni subexpuesto ni sobreexpuesto
        contrast   > 15         # suficiente contraste
    )
    return {"sharpness": sharpness, "brightness": brightness,
            "contrast": contrast,  "quality_ok": quality_ok}


def compute_video_iqs(video_path: str, sample_n: int = 20) -> pd.DataFrame:
    """Calcula IQS para N frames equidistantes de un video."""
    cap    = cv2.VideoCapture(video_path)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs   = np.linspace(0, total - 1, sample_n, dtype=int)
    rows   = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, frame = cap.read()
        if ret:
            row = compute_iqs(frame)
            row['frame_idx'] = int(i)
            rows.append(row)
    cap.release()
    return pd.DataFrame(rows)


## 2.4 Procesar todos los videos

In [ ]:
video_files = sorted(DATA_PATH.glob("*.mp4")) + sorted(DATA_PATH.glob("*.avi"))
iqs_summary = []

for vf in video_files:
    iqs_df = compute_video_iqs(str(vf))
    pct_ok = iqs_df['quality_ok'].mean()
    iqs_summary.append({
        "filename"   : vf.name,
        "pct_ok"     : round(pct_ok, 3),
        "mean_sharp" : round(iqs_df['sharpness'].mean(), 1),
        "mean_bright": round(iqs_df['brightness'].mean(), 1),
        "video_ok"   : pct_ok > 0.5   # > 50% frames aceptables
    })
    print(f"{vf.name:30s}  pct_ok={pct_ok:.2f}")

iqs_df_all = pd.DataFrame(iqs_summary)
iqs_df_all.to_csv(DATA_PATH / "video_iqs.csv", index=False)
print(f"\nVideos aceptables: {iqs_df_all['video_ok'].sum()} / {len(iqs_df_all)}")


## 2.5 Guardar frames preprocesados para anotación

In [ ]:
def extract_best_frames(video_path: str, out_dir: Path, n: int = 5) -> list[str]:
    """
    Extrae los N frames con mayor sharpness para anotación manual.
    Aplica estabilización + CLAHE antes de guardar.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // (n * 5))

    candidates = []
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret: continue
        iqs = compute_iqs(frame)
        if iqs['quality_ok']:
            candidates.append((iqs['sharpness'], i, frame))
    cap.release()

    candidates.sort(reverse=True)
    saved = []
    for rank, (sharp, idx, frame) in enumerate(candidates[:n]):
        corrected = correct_illumination(frame)
        fname = out_dir / f"frame_{idx:05d}.png"
        cv2.imwrite(str(fname), corrected)
        saved.append(str(fname))
    return saved


# Ejecutar para todos los videos válidos
FRAMES_DIR = DATA_PATH / "frames_for_annotation"
for vf in video_files:
    vname = vf.stem
    saved = extract_best_frames(str(vf), FRAMES_DIR / vname, n=5)
    print(f"{vname}: {len(saved)} frames guardados")

print("\nFrames listos para anotar en LabelMe.")
print(f"Directorio: {FRAMES_DIR}")
